# Исследовательский анализ данных (EDA) — GTSDB

Анализ датасета для задачи **детекции дорожных знаков** (43 класса, GTSDB).
Перед запуском выполните подготовку данных:

```bash
python main.py --mode prepare --config configs/default.yaml
```

Ноутбук строит: распределение классов, распределение размеров bounding box,
размеров изображений и показывает примеры с разметкой.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from src.utils.utils import load_config
from src.utils.visualize import draw_detections
sns.set_theme(style='whitegrid')

cfg = load_config('../configs/default.yaml')
processed = Path('..') / cfg['data']['processed']
classes = (processed / 'gtsdb'/ 'classes.txt').read_text(encoding='utf-8').splitlines()
print('Классов:', len(classes))
print(classes)

NameError: name 'gtsdb' is not defined

In [ ]:
# Загружаем манифесты train/val в единый DataFrame (одна строка = один знак)
rows = []
for split in ['train', 'val']:
    recs = json.loads((processed / f'manifest_{split}.json').read_text(encoding='utf-8'))
    for r in recs:
        for (x1, y1, x2, y2), label in zip(r['boxes'], r['labels']):
            rows.append({
                'split': split,
                'class': classes[label],
                'img_w': r['width'], 'img_h': r['height'],
                'box_w': x2 - x1, 'box_h': y2 - y1,
                'box_area_frac': ((x2 - x1) * (y2 - y1)) / (r['width'] * r['height']),
            })
df = pd.DataFrame(rows)
print('Всего знаков (bounding box):', len(df))
print('Изображений:', sum(len(json.loads((processed / f"manifest_{s}.json").read_text(encoding="utf-8"))) for s in ['train', 'val']))
df.head()

## Распределение по классам

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
order = df['class'].value_counts().index
sns.countplot(data=df, y='class', hue='split', order=order, ax=ax)
ax.set(title='Количество знаков по классам', xlabel='кол-во', ylabel='класс')
plt.tight_layout(); plt.show()

## Размеры изображений и bounding box

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='img_w', y='img_h', alpha=0.3, ax=axes[0])
axes[0].set(title='Размеры изображений', xlabel='ширина', ylabel='высота')
sns.histplot(df['box_area_frac'], bins=40, ax=axes[1])
axes[1].set(title='Доля площади bounding box от изображения', xlabel='доля площади')
plt.tight_layout(); plt.show()
print(df[['box_area_frac']].describe())

## Примеры изображений с разметкой
Обратите внимание на небольшой размер знаков относительно кадра —
ключевая сложность детекции на GTSDB (много мелких объектов на изображении).

In [ ]:
recs = json.loads((processed / 'manifest_train.json').read_text(encoding='utf-8'))
samples = recs[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axes.ravel(), samples):
    img_bgr = cv2.imread(r['image'])
    dets = [(tuple(box), 1.0, label) for box, label in zip(r['boxes'], r['labels'])]
    drawn = draw_detections(img_bgr, dets, classes)
    ax.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{len(dets)} знак(ов)"); ax.axis('off')
plt.tight_layout(); plt.show()